# 10 - ربط واستخراج البيانات من قواعد بيانات SQL (SQL Databases Ingestion & RAG)

مرحباً بك في الدرس العاشر والأخير من مسار **Data Ingestion** ضمن كورس **RAG Mastery**!

في هذا الدرس، نتعلم كيفية استخراج البيانات من **قواعد البيانات العلائقية (Relational SQL Databases)**:
1. **بناء قاعدة بيانات تجريبية (`company.db`)**: إنشاء جدولي الموظفين (`employees`) والمشاريع (`projects`) مع علاقات المفاتيح الأجنبية.
2. **أدوات LangChain SQL Utilities (`SQLDatabase`)**: فحص بنية الجداول ومخطط الـ DDL آلياً.
3. **استخراج المستندات عبر `SQLDatabaseLoader`**: تحويل نتائج استعلامات SQL إلى كائنات `Document`.
4. **الربط العلائقي الدلالي (`Cross-Table Relational JOINs`)**: فك التسوية ودمج الجداول المرتبطة في وثائق غنية بالمعلومات لتمكين البحث الدلالي بالمتجهات.

## 1. استيراد المكتبات والتحقق من البيئة

In [ ]:
import os
import sqlite3
from pathlib import Path
from typing import List, Dict, Any

from langchain_core.documents import Document
from langchain_community.utilities import SQLDatabase
from langchain_community.document_loaders import SQLDatabaseLoader

print("SQL Database utilities and Document loaders imported successfully!")

## 2. إنشاء وتغذية قاعدة بيانات SQLite تجريبية (`company.db`)

سنقوم بإنشاء مجلد `data/databases/` وبناء قاعدة بيانات تحوي جدولين مترابطين:
- `employees`: بيانات الموظفين (معرف، اسم، دور، قسم، راتب).
- `projects`: بيانات المشاريع (معرف، اسم، حالة، ميزانية، ومعرف قائد المشروع `lead_id`).

In [ ]:
db_dir = Path("data/databases")
db_dir.mkdir(parents=True, exist_ok=True)
db_file = db_dir / "company.db"

conn = sqlite3.connect(str(db_file))
cur = conn.cursor()

# 1. إنشاء جدول الموظفين
cur.execute("""
CREATE TABLE IF NOT EXISTS employees (
    id INTEGER PRIMARY KEY,
    name TEXT,
    role TEXT,
    department TEXT,
    salary REAL
)
""")

# 2. إنشاء جدول المشاريع
cur.execute("""
CREATE TABLE IF NOT EXISTS projects (
    id INTEGER PRIMARY KEY,
    name TEXT,
    status TEXT,
    budget REAL,
    lead_id INTEGER,
    FOREIGN KEY(lead_id) REFERENCES employees(id)
)
""")

# 3. إدخال السجلات التجريبية
employees_data = [
    (1, "John Doe", "Senior Developer", "Engineering", 95000.0),
    (2, "Jane Smith", "Lead Data Scientist", "Data Science", 115000.0),
    (3, "Mike Johnson", "DevOps Architect", "Infrastructure", 105000.0),
    (4, "Sarah Williams", "Frontend Specialist", "Engineering", 85000.0)
]
cur.executemany("INSERT OR REPLACE INTO employees VALUES (?, ?, ?, ?, ?)", employees_data)

projects_data = [
    (1, "RAG Knowledge Base", "In Progress", 50000.0, 1),
    (2, "Customer Support AI Agent", "Completed", 75000.0, 2),
    (3, "Kubernetes Cloud Migration", "In Progress", 120000.0, 3),
    (4, "Predictive Analytics Engine", "Planning", 90000.0, 2)
]
cur.executemany("INSERT OR REPLACE INTO projects VALUES (?, ?, ?, ?, ?)", projects_data)

conn.commit()
conn.close()
print(f"[+] SQLite Database created successfully at: {db_file}")

## 3. الطريقة الأولى: استكشاف وفحص قاعدة البيانات عبر `SQLDatabase` Utility

تُستخدم `SQLDatabase` لفحص الجداول المتاحة ومخطط الـ DDL تلقائياً.

In [ ]:
db_uri = f"sqlite:///{db_file.as_posix()}"
db = SQLDatabase.from_uri(db_uri)

print(f"[+] Usable Tables: {db.get_usable_table_names()}")
print("\n--- Employees Table Schema & Sample Records ---")
print(db.get_table_info(["employees"]))

print("--- Projects Table Schema & Sample Records ---")
print(db.get_table_info(["projects"]))

## 4. الطريقة الثانية: استخراج السجلات عبر `SQLDatabaseLoader`

تحويل نتائج استعلام SQL مباشر إلى كائنات `Document` جاهزة للتقطيع والتضمين.

In [ ]:
sql_query = "SELECT name, role, department, salary FROM employees WHERE salary >= 95000"

loader = SQLDatabaseLoader(
    query=sql_query,
    db=db
)
query_docs = loader.load()

print(f"[+] Extracted {len(query_docs)} records from SQL query:\n")
for i, doc in enumerate(query_docs, 1):
    print(f"--- Record #{i} ---")
    print(doc.page_content)
    print()

## 5. الطريقة الثالثة: فحص المخطط الشامل وتوليد وثائق الجداول (`sql_to_documents`)

استخراج المخطط الهيكلي لكل جدول وعينات من صفوفه كوثيقة سياقية موجهة للـ LLM.

In [ ]:
def sql_to_documents(db_path: str) -> List[Document]:
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    documents: List[Document] = []

    # 1. استخراج أسماء الجداول من sqlite_master
    cur.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = [t[0] for t in cur.fetchall()]

    for table_name in tables:
        # 2. استخراج بيانات الأعمدة عبر PRAGMA table_info
        cur.execute(f"PRAGMA table_info({table_name});")
        col_info = cur.fetchall()
        columns = [c[1] for c in col_info]

        # 3. استخراج الصفوف
        cur.execute(f"SELECT * FROM {table_name};")
        rows = cur.fetchall()

        # صياغة محتوى وصفي للجدول
        content_lines = [
            f"Database Table: {table_name}",
            f"Defined Columns: {', '.join(columns)}",
            f"Total Rows: {len(rows)}",
            "Sample Data:"
        ]
        for row in rows[:5]:
            row_str = " | ".join([f"{col}: {val}" for col, val in zip(columns, row)])
            content_lines.append(f"  - {row_str}")

        metadata = {
            "source": Path(db_path).name,
            "table_name": table_name,
            "row_count": len(rows),
            "columns_count": len(columns)
        }
        documents.append(Document(page_content="\n".join(content_lines), metadata=metadata))

    conn.close()
    return documents

table_docs = sql_to_documents(str(db_file))
print(f"[+] Generated {len(table_docs)} table schema documents.\n")
for doc in table_docs:
    print(f"=== Table: {doc.metadata['table_name']} ===")
    print(doc.page_content)
    print(f"Metadata: {doc.metadata}\n")

## 6. الطريقة الرابعة: ربط العلاقات عبر الجداول (`Cross-Table Relational JOIN Serialization`)

نظراً لأن قواعد البيانات المتجهة (`Vector DBs`) لا تدعم عمليات `JOIN` برمجياً، فإن أفضل أسلوب هو **دمج العلاقات دلالياً** عبر استعلام SQL يربط الجداول ببعضها ويحول كل علاقة إلى فقرة مفهومة.

In [ ]:
def serialize_relational_data(db_path: str) -> List[Document]:
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()

    query = """
    SELECT 
        e.name AS lead_name,
        e.role AS lead_role,
        e.department,
        p.name AS project_name,
        p.status AS project_status,
        p.budget AS project_budget
    FROM employees e
    INNER JOIN projects p ON e.id = p.lead_id
    """
    cur.execute(query)
    rows = cur.fetchall()
    documents: List[Document] = []

    for row in rows:
        lead_name, role, dept, proj_name, status, budget = row

        # صياغة نص متكامل للعلاقة
        content = (
            f"Project Overview: {proj_name}\n"
            f"Current Status: {status}\n"
            f"Approved Budget: ${budget:,.2f}\n"
            f"Project Lead: {lead_name} (Position: {role})\n"
            f"Managing Department: {dept}"
        )

        metadata = {
            "source": Path(db_path).name,
            "relation_type": "employee_leads_project",
            "project_name": proj_name,
            "project_status": status,
            "budget": budget,
            "lead_name": lead_name,
            "department": dept
        }
        documents.append(Document(page_content=content, metadata=metadata))

    conn.close()
    return documents

relational_documents = serialize_relational_data(str(db_file))
print(f"[+] Generated {len(relational_documents)} relational joined documents:\n")
for doc in relational_documents:
    print(f"--- {doc.metadata['project_name']} [{doc.metadata['project_status']}] ---")
    print(doc.page_content)
    print(f"Metadata: {doc.metadata}\n")

## 7. الخلاصة والمقارنة المعمارية

| الاستراتيجية | الأنسب لـ | المزايا | الملاحظات |
| :--- | :--- | :--- | :--- |
| **`SQLDatabase Utility`** | استكشاف بنية المخطط | يستخرج أوامر DDL وعينات البيانات تلقائياً | مخصص لسياق النماذج وليس للـ Chunks مباشرة |
| **`SQLDatabaseLoader`** | استخراج نتائج استعلامات محددة | مرن وسريع ويقبل استعلامات SQL مخصصة | يحتاج كتابة استعلام SELECT مسبقاً |
| **`Relational JOIN Serialization`** | قواعد البيانات المتعددة الجداول | يحل مشكلة عجز Vector DB عن الـ JOIN | يتطلب فهم العلاقات بين الجداول مسبقاً |